In [8]:
import sqlite3
import os
import re

conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

BOOK_IDS = {
    "Mt": 47,
    "Mk": 48,
    "Lk": 49,
    "Jn": 50,
}

def parse_filename(filename):
    name = os.path.splitext(filename)[0].replace("_en", "")
    match = re.match(r"([A-Za-z]+)\_(\d+)", name)
    if not match:
        raise ValueError(f"文件名格式错误：{filename}")

    book_abbr = match.group(1)
    chapter = int(match.group(2))
    book_id = BOOK_IDS[book_abbr]
    return book_abbr, chapter, book_id


def import_verses_from_file(filepath):
    book_abbr, chapter, book_id = parse_filename(os.path.basename(filepath))

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for i, text_en in enumerate(lines, start=1):
        verse_id = f"{book_abbr}.{chapter}.{i}"

        cur = cursor.execute(
            "SELECT text_en FROM verse WHERE id = ?", (verse_id,)
        )
        row = cur.fetchone()

        if row is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, text_en, ""))
            print(f"➕ 新增 {verse_id}")

        elif row[0] == text_en:
            pass

        else:
            ans = input(f"\n⚠️  已存在 {verse_id}，是否覆盖？(Y/N): ").strip().upper()
            if ans == "Y":
                cursor.execute(
                    "UPDATE verse SET text_en = ? WHERE id = ?",
                    (text_en, verse_id)
                )
                print(f"♻️  已覆盖 {verse_id}")
            else:
                print(f"⏭️  跳过 {verse_id}")

    conn.commit()
    print(f"\n✅ {book_abbr}.{chapter} 导入完成")


# 只问文件名
if __name__ == "__main__":
    filename = input("请输入文件名（如 Mt.1.txt）：").strip()
    filepath = os.path.join("outputs", "plaintext", filename)

    if not os.path.exists(filepath):
        print(f"❌ 文件不存在：{filepath}")
    else:
        import_verses_from_file(filepath)

请输入文件名（如 Mt.1.txt）：Mt_1_en.txt
➕ 新增 Mt.1.1
➕ 新增 Mt.1.2
➕ 新增 Mt.1.3
➕ 新增 Mt.1.4
➕ 新增 Mt.1.5
➕ 新增 Mt.1.6
➕ 新增 Mt.1.7
➕ 新增 Mt.1.8
➕ 新增 Mt.1.9
➕ 新增 Mt.1.10
➕ 新增 Mt.1.11
➕ 新增 Mt.1.12
➕ 新增 Mt.1.13
➕ 新增 Mt.1.14
➕ 新增 Mt.1.15
➕ 新增 Mt.1.16
➕ 新增 Mt.1.17
➕ 新增 Mt.1.18
➕ 新增 Mt.1.19
➕ 新增 Mt.1.20
➕ 新增 Mt.1.21
➕ 新增 Mt.1.22
➕ 新增 Mt.1.23
➕ 新增 Mt.1.24
➕ 新增 Mt.1.25

✅ Mt.1 导入完成


In [9]:
import sqlite3
import os
import re

# ==========================
# 数据库连接
# ==========================
conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

# ==========================
# 书卷缩写 → book_id 映射（思高 / NRSVCE 按需调整）
# ==========================
BOOK_IDS = {
    "Mt": 47,
    "Mk": 48,
    "Lk": 49,
    "Jn": 50,
}

# ==========================
# 解析文件名：Mt.1.txt
# ==========================
def parse_filename(filename):
    name = os.path.splitext(filename)[0]
    match = re.match(r"([A-Za-z]+)\_(\d+)", name)
    if not match:
        raise ValueError(f"文件名格式错误：{filename}")

    book_abbr = match.group(1)
    chapter = int(match.group(2))
    book_id = BOOK_IDS[book_abbr]
    return book_abbr, chapter, book_id

# ==========================
# 导入中文经文到 text_cn 列
# 只判断 text_cn 是否为空
# ==========================
def import_verses_from_file(filepath):
    book_abbr, chapter, book_id = parse_filename(os.path.basename(filepath))

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for i, text_cn in enumerate(lines, start=1):
        verse_id = f"{book_abbr}.{chapter}.{i}"

        cur = cursor.execute(
            "SELECT text_cn FROM verse WHERE id = ?", (verse_id,)
        )
        row = cur.fetchone()

        # verse 不存在（极端情况）
        if row is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, "", text_cn))
            print(f"➕ 新增 {verse_id}")

        # text_cn 为空 → 直接写入
        elif row[0] is None or row[0] == "":
            cursor.execute(
                "UPDATE verse SET text_cn = ? WHERE id = ?",
                (text_cn, verse_id)
            )
            print(f"➕ 写入 text_cn: {verse_id}")

        # text_cn 已有内容 → 询问是否覆盖
        else:
            ans = input(f"\n⚠️  已存在 {verse_id}，是否覆盖中文译文？(Y/N): ").strip().upper()
            if ans == "Y":
                cursor.execute(
                    "UPDATE verse SET text_cn = ? WHERE id = ?",
                    (text_cn, verse_id)
                )
                print(f"♻️  已覆盖 text_cn: {verse_id}")
            else:
                print(f"⏭️  跳过 {verse_id}")

    conn.commit()
    print(f"\n✅ {book_abbr}.{chapter} 中文经文导入完成")

# ==========================
# 主程序
# ==========================
if __name__ == "__main__":
    filename = input("请输入中文经文文件名（如 Mt.1.txt）：").strip()
    filepath = os.path.join("outputs", "plaintext", filename)

    if not os.path.exists(filepath):
        print(f"❌ 文件不存在：{filepath}")
    else:
        import_verses_from_file(filepath)

请输入中文经文文件名（如 Mt.1.txt）：Mt_1_cn.txt
➕ 写入 text_cn: Mt.1.1
➕ 写入 text_cn: Mt.1.2
➕ 写入 text_cn: Mt.1.3
➕ 写入 text_cn: Mt.1.4
➕ 写入 text_cn: Mt.1.5
➕ 写入 text_cn: Mt.1.6
➕ 写入 text_cn: Mt.1.7
➕ 写入 text_cn: Mt.1.8
➕ 写入 text_cn: Mt.1.9
➕ 写入 text_cn: Mt.1.10
➕ 写入 text_cn: Mt.1.11
➕ 写入 text_cn: Mt.1.12
➕ 写入 text_cn: Mt.1.13
➕ 写入 text_cn: Mt.1.14
➕ 写入 text_cn: Mt.1.15
➕ 写入 text_cn: Mt.1.16
➕ 写入 text_cn: Mt.1.17
➕ 写入 text_cn: Mt.1.18
➕ 写入 text_cn: Mt.1.19
➕ 写入 text_cn: Mt.1.20
➕ 写入 text_cn: Mt.1.21
➕ 写入 text_cn: Mt.1.22
➕ 写入 text_cn: Mt.1.23
➕ 写入 text_cn: Mt.1.24
➕ 写入 text_cn: Mt.1.25

✅ Mt.1 中文经文导入完成


In [3]:
# 清空数据

import sqlite3

DB_PATH = "db/bible.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("DELETE FROM verse;")
conn.commit()
conn.close()

print("✅ verse 表数据已全部清空")

✅ verse 表数据已全部清空
